# 13. League History and Records

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order. Run `12_backtesting.ipynb` first.

`04_capturing_a_league.ipynb` persisted six real seasons of standings, matchups, and transactions. `nuclearff.archive` mines that same stored data for a completely different purpose than valuation -- a league's own record book: champions, blowouts, rivalries, draft steals and busts, and a full career profile per manager. Every function here is read/aggregate-only: no new Sleeper fetching, just real history already on disk. Mirrors the [League History](https://nolmacdonald.github.io/nuclearff/tutorial/13_league_history.html) docs page.

## Championship history

`championship_history` is the shared foundation the rest of this notebook joins against -- one row per season: champion, runner-up, and regular-season leader:

In [1]:
import polars as pl

from nuclearff.archive.champions import championship_history
from nuclearff.duckdb_io import read_table

db_path = "./demo/data/cache/nuclearff.duckdb"
standings = read_table(db_path, "sleeper_standings")

champs = championship_history(standings)
champs.sort("season").select(
    [
        "season",
        "champion_display_name",
        "runner_up_display_name",
        "regular_season_leader_display_name",
    ]
)

season,champion_display_name,runner_up_display_name,regular_season_leader_display_name
i64,str,str,str
2021,"""bigTETONclimber""","""casitzmann""","""aperry151"""
2022,null,"""nolmacdonald""",null
2023,"""ksavabi""","""hyoga10""","""hyoga10"""
2024,"""aperry151""","""ksavabi""","""casitzmann"""
2025,"""nolmacdonald""","""ksavabi""","""casitzmann"""
2026,null,null,"""jwhitney0220"""


Watch for real, honest nulls in both directions: a season can show a null champion because that season's champion roster's owner isn't resolvable in that season's user list (a real Sleeper data quirk `14_trade_network.ipynb` runs into for trades too), and the current in-progress season has no champion yet -- `championship_history` doesn't guess at an unfinished season.

## Season awards

`season_awards` surfaces three superlatives, each honestly derivable from `sleeper_standings` alone -- no subjective judgment call invented for something this project has no data to back (a "best trade" or "most exciting week" award would need a real, stated scoring rule from you, not a guessed heuristic):

In [2]:
from nuclearff.archive.awards import season_awards

awards = season_awards(standings)
awards

kind,season,manager,value
str,i64,str,f64
"""best_regular_season""",2025,"""casitzmann""",2055.82
"""cinderella_run""",2024,"""aperry151""",4.0
"""biggest_improvement""",2023,"""hyoga10""",402.06


Three real, spot-checkable kinds of result to look for:

- **Best regular season** -- the most dominant #1 seed in this league's history by total points, among every season's own regular-season leader.
- **Cinderella run** -- a champion whose real regular-season rank was well outside the top spot (`value` is the rank gap). Since a champion's `final_rank` is always 1 by definition, this needs no playoff-week boundary at all -- unlike `14_trade_network.ipynb`'s rivalries split, "did this team overcome a bad regular season" is already answered by `regular_season_rank` vs. `final_rank` alone.
- **Biggest year-over-year improvement** -- the largest single-manager point jump between **consecutive** seasons. A gap season (absent one year, back the next) is not treated as "improvement."

An award with no eligible season (e.g. no season has a resolved `final_rank` yet) is simply absent from the result rather than a row of nulls.

## Records: extremes and streaks

`score_extremes` and `margin_extremes` scan every real matchup for the league's all-time highest/lowest score and biggest blowout/closest margin:

In [3]:
from nuclearff.archive.records import longest_streak, margin_extremes, score_extremes
from nuclearff.sleeper.wins import weekly_results

matchups = read_table(db_path, "sleeper_matchups")
score_extremes(matchups, standings)

kind,manager,season,week,points,opponent
str,str,i64,i64,f64,str
"""highest""","""thatbolb""",2022,2,206.41,"""hyoga10"""
"""lowest""","""casitzmann""",2026,3,0.0,null


In [4]:
margin_extremes(matchups, standings)

kind,season,week,winner,winner_points,loser,loser_points,margin
str,i64,i64,str,f64,str,f64,f64
"""biggest_blowout""",2022,2,"""thatbolb""",206.41,"""hyoga10""",100.78,105.63
"""closest_margin""",2024,2,"""hyoga10""",140.22,"""jwhitney0220""",140.16,0.06


`longest_streak` needs each roster's real per-week win/loss, from `weekly_results` (`15_wins_and_leagues.ipynb` uses the same function for cumulative wins):

In [5]:
results = weekly_results(matchups)
win_streaks = longest_streak(results, standings, "win")
win_streaks.sort("length", descending=True).head(3)

manager,streak_type,length,start_season,start_week,end_season,end_week
str,str,u32,i64,i64,i64,i64
"""casitzmann""","""win""",5,2024,1,2025,2
"""thatbolb""","""win""",5,2021,2,2022,3
"""ruhbberduhcky""","""win""",4,2024,1,2025,1


Pass `"loss"` for `streak_type` to get the same for losing streaks instead.

## Rivalries: head-to-head

`head_to_head` returns the all-time combined record for every manager pair that has ever met:

In [6]:
from nuclearff.archive.rivalries import head_to_head

h2h = head_to_head(matchups, standings)
h2h.sort("games", descending=True).select(
    ["manager_a", "manager_b", "games", "wins_a", "wins_b", "biggest_blowout_margin"]
).head(3)

manager_a,manager_b,games,wins_a,wins_b,biggest_blowout_margin
str,str,u32,u32,u32,f64
"""hyoga10""","""thatbolb""",4,3,1,105.63
"""aperry151""","""nolmacdonald""",4,2,2,22.14
"""casitzmann""","""hyoga10""",4,3,1,14.62


This is the combined regular-season **and** playoff record. `head_to_head_by_phase` splits it -- using `LeagueConfig`'s real `playoff_week_start` (`02_configuration.ipynb`) to tell the two apart, rather than a hardcoded week range:

In [7]:
from nuclearff.archive.rivalries import head_to_head_by_phase

league_configs = read_table(db_path, "sleeper_league_configs")
split = head_to_head_by_phase(matchups, standings, league_configs)

top_row = h2h.sort("games", descending=True).row(0, named=True)
top_pair = split.filter(
    (pl.col("manager_a") == top_row["manager_a"])
    & (pl.col("manager_b") == top_row["manager_b"])
)
top_pair.select(["phase", "games", "wins_a", "wins_b"])

phase,games,wins_a,wins_b
str,u32,u32,u32
"""regular_season""",4,3,1


The two phases' `games` should sum to exactly this pair's combined `games` above -- the real cross-check `head_to_head_by_phase`'s own docstring promises. A `(league_id, season)` with no resolvable `playoff_week_start` -- a real "Chopped" league with no bracket, or a season simply missing from `league_configs` -- is excluded from **both** phases rather than guessed into one.

In [8]:
top_pair["games"].sum() == top_row["games"]

True

## Draft retrospectives

`grade_historical_picks` grades every historical pick against how the player *actually* performed that season -- a real final-output number, not a pre-draft projection. It needs `season_actuals` (exploded weekly box scores) concatenated across every season on file:

In [9]:
from nuclearff.archive.draft_retro import grade_historical_picks
from nuclearff.sleeper.performance import season_actuals

picks = read_table(db_path, "sleeper_draft_picks")
actuals = pl.concat(
    [
        season_actuals(matchups, standings, lid)
        for lid in standings["league_id"].unique()
    ],
    how="diagonal_relaxed",
)
graded = grade_historical_picks(picks, actuals, standings).filter(
    pl.col("season") < 2026
)

graded.sort("rank_delta", descending=True).select(
    [
        "season",
        "manager",
        "player_name",
        "round",
        "pick_no",
        "expected_rank",
        "actual_rank",
        "rank_delta",
    ]
).head(3)

season,manager,player_name,round,pick_no,expected_rank,actual_rank,rank_delta
i64,str,str,i64,i64,u32,u32,i64
2023,"""hyoga10""","""Russell Wilson""",15,145,145,10,135
2025,"""jwhitney0220""","""Jared Goff""",14,140,140,7,133
2025,"""thatbolb""","""Aaron Rodgers""",15,147,147,16,131


A large positive `rank_delta` is a real steal: a player picked late by draft position who actually finished as one of the league's best that season. Sort ascending instead for busts -- a top pick who finished far down the real rankings, often due to a real injury. `sleeper_draft_picks.is_keeper` is a known, unresolved confound here -- a keeper pick's draft slot reflects a league rule, not a real assessment of the player's value, and this function leaves filtering it to the caller rather than silently excluding keeper leagues' data.

> **Why this matters:** "draft steal" and "draft bust" are fun to read, but the `is_keeper` caveat above is the kind of thing that quietly invalidates a whole draft-retro report if you skip it. A keeper kept at round 14 because a league rule says so isn't the same signal as a real 14th-round gamble that happened to pay off -- filter `is_keeper` out before drawing conclusions about draft *skill* specifically, and keep it in only if you're asking a different question ("which picks, keeper or not, produced the most value").

## Franchise profiles

`franchise_profile` assembles everything above into one row per manager -- pure assembly, no new stat computed:

In [10]:
from nuclearff.archive.franchise import franchise_profile
from nuclearff.sleeper import draft_order_stats
from nuclearff.sleeper.trades import load_trades, manager_trade_counts

se, me = score_extremes(matchups, standings), margin_extremes(matchups, standings)
win_streaks = longest_streak(results, standings, "win")
loss_streaks = longest_streak(results, standings, "loss")
trade_counts = manager_trade_counts(load_trades(db_path))
draft_stats = draft_order_stats(picks, standings)

profile = franchise_profile(
    standings, champs, trade_counts, draft_stats, se, me, win_streaks, loss_streaks
)
profile.sort("career_wins", descending=True).select(
    [
        "manager",
        "seasons_played",
        "career_wins",
        "career_losses",
        "championships",
        "records_held",
    ]
).head(4)

manager,seasons_played,career_wins,career_losses,championships,records_held
str,u32,i64,i64,u32,list[str]
"""casitzmann""",6,62,40,0,"[""lowest_single_week_score"", ""longest_win_streak""]"
"""hyoga10""",6,56,46,0,"[""closest_margin_win""]"
"""thatbolb""",6,54,48,0,"[""highest_single_week_score"", ""biggest_blowout""]"
"""aperry151""",6,53,49,1,[]


A common, real-feeling fantasy football story that a plain career-record table would hide: check whether this league's own career wins leader has actually won a championship. `championships` sitting right next to career record answers that instantly instead of needing a separate lookup.

## Team name history

`team_name_changes` tracks each manager's chronological team-name history, keyed by `owner_id` (stable across name changes) rather than the display name that changes with it:

In [11]:
from nuclearff.archive.name_history import team_name_changes
from nuclearff.sleeper import SleeperClient, walk_league_chain
from nuclearff.sleeper.roster_names import fetch_and_write_team_names

league_id = "1367225133634191360"
with SleeperClient(cache_dir="./demo/data/cache") as client:
    chain = walk_league_chain(client, league_id, max_seasons=20)
    fetch_and_write_team_names(client, chain, db_path)

roster_names = read_table(db_path, "sleeper_roster_names")
names_history = team_name_changes(roster_names, standings)
names_history.sort("change_count", descending=True).head(3)

owner_id,manager,name_sequence,change_count
str,str,list[null],u32
"""332632476830679040""","""nolmacdonald""","[null, null, … null]",0
"""451502081426059264""","""aperry151""","[null, null, … null]",0
"""460983681738076160""","""bigTETONclimber""",[null],0


A league where nobody has ever set a custom Sleeper team name will show an honest `change_count` of 0 across the board -- every roster falls back to a null name every season. A league that *does* use custom team names shows real sequences and non-zero counts here instead.

## Waiver spend

`manager_waiver_spend` and `career_waiver_spend` turn stored transactions into real FAAB spending history:

In [12]:
from nuclearff.archive.waiver_spend import career_waiver_spend, manager_waiver_spend

transactions = read_table(db_path, "sleeper_transactions")
spend = manager_waiver_spend(transactions)
career = career_waiver_spend(spend)
career.sort("total_spent", descending=True).head(3)

manager,total_spent,players_acquired,failed_claims,avg_cost_per_player
str,i64,u32,u32,f64
"""nawfeastdallas""",211,18,15,11.722222
"""nolmacdonald""",204,18,15,11.333333
"""casitzmann""",137,19,34,7.210526


A manager's real failed-claims count sitting next to their successful acquisitions is the kind of detail a season-total spend number alone would hide -- someone who spent the most may have also lost the most bids along the way.

## On this day

`transactions_on_this_day` finds every real transaction whose anniversary matches a given date -- `transaction_summary_rows` turns that into display-ready text, resolving player ids to names when a `sleeper_players` table is available:

In [13]:
from datetime import date

from nuclearff.archive.on_this_day import (
    transaction_summary_rows,
    transactions_on_this_day,
)
from nuclearff.ids import read_sleeper_players

players = read_sleeper_players(db_path)
today = date.today()
on_this_day = transactions_on_this_day(transactions, today)
transaction_summary_rows(on_this_day, players).head(3)

year,type,parties,summary,group_id
i64,str,str,str,str
2024,"""waiver""","""nawfeastdallas""","""added DeAndre Hopkins""","""1143746753117077504"""
2024,"""free_agent""","""nolmacdonald""","""added Darnell Mooney""","""1143680342705008640"""
2024,"""free_agent""","""thatbolb""","""added Brenton Strange dropped …","""1143622397124411392"""


Since this cell uses today's real calendar date, its output will differ every time this notebook is re-executed -- that's expected, not a bug; it's the whole point of an "on this day" lookup.

## What's Next

League history above was keyed by manager and season. `14_trade_network.ipynb` zooms into one specific relationship -- who trades with whom -- as ten different visualizations from the same stored transaction data.

## See Also

- `04_capturing_a_league.ipynb` -- where every table this chapter reads comes from.
- [League History](https://nolmacdonald.github.io/nuclearff/tutorial/13_league_history.html) -- the matching docs page.